# Домашняя работа 3

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

pandas 2.2.2 (6830, 7)


## Исходники домашней работы

In [2]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei": "string"})
lookup = organizations[org_keys + ["organization_name"]].copy()
assert not lookup.duplicated(org_keys).any()
print(organizations.head(3))

                    lei                       organization_name  source_count  \
0  549300AG64NHILB7ZP05                      LOANDEPOT.COM, LLC           114   
1  QFROUN1UWUYU0DVIWD51  Fifth Third Bank, National Association             3   
2  5493003GQDUH26DNNH17               NAVY FEDERAL CREDIT UNION           742   

   activity_year  
0           2023  
1           2023  
2           2023  


## H1 Профиль отказов

In [3]:
prepared = decisions.copy()
prepared["is_denied"] = prepared["action_taken"].eq(3)
denial = prepared.groupby(org_keys).agg(
    n_decisions=("action_taken", "size"),
    n_denied=("is_denied", "sum"),
).reset_index()
denial["denial_rate"] = denial["n_denied"] / denial["n_decisions"]

denial = denial.merge(lookup, on=org_keys, how="left", validate="one_to_one", indicator=True)
assert denial["_merge"].eq("both").all()
assert denial["organization_name"].notna().all()
assert denial["n_decisions"].sum() == len(decisions)
assert denial["n_denied"].sum() == decisions["action_taken"].eq(3).sum()
assert denial["denial_rate"].between(0, 1).all()

denial = denial.drop(columns="_merge")
denial.to_csv(WORK / "home_denial_profile.csv", index=False)
print("Всего решений", denial["n_decisions"].sum())
print("Всего отказов", denial["n_denied"].sum())
denial.head(5)

Всего решений 6830
Всего отказов 691


,lei,activity_year,n_decisions,n_denied,denial_rate,organization_name
0,03D0JEWFDFUS0SEEKG89,2023,91,15,0.164835,"TD Bank, National Association"
1,1IE8VN30JCEQV1H4R804,2023,7,2,0.285714,"HSBC BANK USA, NATIONAL ASSOCIATION"
2,213800KVD76WKI6PN868,2023,2,1,0.500000,"CAPITAL CENTER, L.L.C."
3,213800QUAI2VH5YM6310,2023,89,9,0.101124,"EMBRACE HOME LOANS, INC."
4,213800XR2TCBQJSF1X93,2023,15,3,0.200000,NBKC BANK


## H2 Группы для просмотра

In [4]:
large_groups = denial.loc[denial["n_decisions"].ge(20)].copy()
large_groups.to_csv(WORK / "home_denial_groups20.csv", index=False)

denial_rate = denial["n_denied"].sum() / denial["n_decisions"].sum()
mean_rate = denial["denial_rate"].mean()
print("Всего групп", len(denial))
print("Групп с 20 решениями и больше", len(large_groups))
print("Решений в отобранных группах", large_groups["n_decisions"].sum())
print("Отказов в отобранных группах", large_groups["n_denied"].sum())
print("Общая доля отказов, %", round(denial_rate * 100, 2))
print("Среднее долей организаций, %", round(mean_rate * 100, 2))
large_groups.head(5)

Всего групп 314
Групп с 20 решениями и больше 61
Решений в отобранных группах 5768
Отказов в отобранных группах 512
Общая доля отказов, % 10.12
Среднее долей организаций, % 14.22


,lei,activity_year,n_decisions,n_denied,denial_rate,organization_name
0,03D0JEWFDFUS0SEEKG89,2023,91,15,0.164835,"TD Bank, National Association"
3,213800QUAI2VH5YM6310,2023,89,9,0.101124,"EMBRACE HOME LOANS, INC."
9,2549003Z8R5QM0WPS603,2023,24,0,0.000000,IDB Global Federal Credit Union
14,2549006II76YXSS5XM65,2023,193,7,0.036269,"George Mason Mortgage, LLC"
26,254900FMBXF85WQYV433,2023,20,1,0.050000,Amalgamated Bank


## H3 Длинная, широкая и снова длинная форма

In [5]:
long = decisions.groupby(long_keys, dropna=False).size().rename("n").reset_index()
wide = long.pivot(index=org_keys, columns="action_taken", values="n")
wide = wide.reindex(columns=[1, 2, 3]).fillna(0).astype("int64")
wide = wide.rename_axis(None, axis=1).reset_index()

restored = wide.melt(
    id_vars=org_keys, value_vars=[1, 2, 3], var_name="action_taken", value_name="n"
)
restored["action_taken"] = restored["action_taken"].astype("int64")
observed = restored.loc[restored["n"].gt(0), long_keys + ["n"]]
observed = observed.sort_values(long_keys).reset_index(drop=True)
long_sorted = long.sort_values(long_keys).reset_index(drop=True)

assert_frame_equal(observed, long_sorted)
assert observed["n"].sum() == len(decisions)
print("Все сочетания", len(restored))
print("Нулевые сочетания", restored["n"].eq(0).sum())
print("Наблюдаемые сочетания", len(observed))
print("Сумма n", observed["n"].sum())

Все сочетания 942
Нулевые сочетания 442
Наблюдаемые сочетания 500
Сумма n 6830


## H+ Безопасное соединение

In [6]:
def safe_enrich(source, names):
    if source[org_keys].isna().any().any():
        raise ValueError("Неизвестный ключ слева")
    if names[org_keys].isna().any().any():
        raise ValueError("Неизвестный ключ справа")
    if names.duplicated(org_keys).any():
        raise ValueError("Повторный ключ справа")

    enriched = source.merge(
        names, on=org_keys, how="left", validate="many_to_one", indicator=True
    )
    assert_frame_equal(
        enriched[source.columns], source.reset_index(drop=True), check_exact=True
    )
    unmatched = enriched.loc[enriched["_merge"].eq("left_only"), source.columns].copy()
    return enriched, unmatched


full_lookup = lookup.copy()
partial_lookup = lookup.iloc[1:].copy()
replayed_lookup = pd.concat([lookup, lookup.iloc[[0]]], ignore_index=True)

changed_name = lookup.iloc[[0]].copy()
changed_name["organization_name"] = "Другое имя"
conflict_lookup = pd.concat([lookup, changed_name], ignore_index=True)
name_counts = conflict_lookup.groupby(org_keys)["organization_name"].nunique(dropna=False)
conflict_keys = name_counts.loc[name_counts.gt(1)]
assert len(conflict_keys) == 1
print("Ключи с разными именами")
print(conflict_keys)

full_enriched, full_unmatched = safe_enrich(decisions, full_lookup)
assert len(full_enriched) == len(decisions)
assert full_unmatched.empty
print("Полный справочник, строк без имени", len(full_unmatched))

partial_enriched, partial_unmatched = safe_enrich(decisions, partial_lookup)
first_org = decisions["lei"].eq(lookup.loc[0, "lei"])
first_year = decisions["activity_year"].eq(lookup.loc[0, "activity_year"])
assert partial_unmatched["source_row"].tolist() == decisions.loc[
    first_org & first_year, "source_row"
].tolist()
assert len(partial_unmatched) > 0
assert len(partial_enriched) == len(decisions)
print("Неполный справочник, строк без имени", len(partial_unmatched))

unknown_source = decisions.copy()
unknown_source.loc[0, "lei"] = pd.NA
unknown_lookup = lookup.copy()
unknown_lookup.loc[0, "lei"] = pd.NA

checks = [
    (decisions, replayed_lookup, "Повторная доставка", "Повторный ключ справа"),
    (decisions, conflict_lookup, "Конфликт имён", "Повторный ключ справа"),
    (unknown_source, lookup, "Пропуск слева", "Неизвестный ключ слева"),
    (decisions, unknown_lookup, "Пропуск справа", "Неизвестный ключ справа"),
]
for source, names, label, expected_error in checks:
    try:
        safe_enrich(source, names)
    except ValueError as error:
        assert str(error) == expected_error
        print(label, "-", error)
    else:
        raise AssertionError("Ожидалась остановка JOIN")

Ключи с разными именами
lei                   activity_year
549300AG64NHILB7ZP05  2023             2
Name: organization_name, dtype: int64
Полный справочник, строк без имени 0
Неполный справочник, строк без имени 25
Повторная доставка - Повторный ключ справа
Конфликт имён - Повторный ключ справа
Пропуск слева - Неизвестный ключ слева
Пропуск справа - Неизвестный ключ справа
